# 03 - PHASE 0 GATE: does the efficiency structure live at the CLASS level?

AGENTS.md Sec 5. The evidence that motivated this project came from synthetic
inject-then-recover, which is circular. It must replicate on REAL logits before Phase 1.

Three mechanisms, all using the SAME conformal offset estimator on abundant data, differing only
in what INDEXES the correction:

1. a single global **temperature** (1 parameter),
2. a per-sample offset indexed by **free energy** `E(x) = -logsumexp(logits)` (n_bins),
3. a per-**class** offset (K parameters).

**PASS (Sec 5):** (3) closes a gap substantially larger than (1) and (2), with non-overlapping CIs.

## Metric: Amendment 3 (approved)

Average set size required to reach **worst-class coverage >= 1-alpha**, every per-group correction
estimated OUT OF SAMPLE. Average set size at nominal MARGINAL coverage was abandoned: marginal
split-CP is already optimal for marginal coverage, so a class-indexed mechanism cannot win on it -
a test that cannot return a positive is not a test. Verified controls: structure present +40.89,
structure absent -2.52.

## Sec 5 demands ABUNDANT data - which on Pl@ntNet means the HEAD

Sec 5 says the per-class offset is fit on *data berlimpah (bukan pada budget realistis)*. Pl@ntNet's
cal split has a **median of 2 samples per class** and 105 classes with none, so on the full label
space the class mechanism would be measuring estimation noise, exactly as it did on CIFAR-100
(~50 cal/class gave +3.56 where 1000/class gave +40.89).

So the PRIMARY run restricts the whole problem to classes with at least `MIN_CAL_PER_CLASS`
calibration samples - the abundant-data regime the section asks for. Restriction is applied to
samples AND score columns (`restrict_to_classes`), so there is no measured-vs-unmeasured asymmetry
to exploit. The unrestricted run is reported as a SECONDARY view, and the two are never merged.

**Scores are OURS**, not LTC's released scores - the checkpoint gate did not pass
(reports/phase0_checkpoint_gate.md). Every conclusion here inherits that.


## 1. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

DATASET    = 'plantnet'
BACKBONE   = 'resnet50_ltc'
CAL_SPLIT  = 'cal'          # calibration/eval pool (OUR scores)

ALPHAS     = (0.05, 0.1)    # alpha=0.01 needs n>=99 per class: only 57/1081 classes
MIN_CAL_PER_CLASS = 50      # PRIMARY: the abundant-data regime Sec 5 asks for
N_SPLITS   = 50             # random cal/eval splits (Sec 8.4 asks >=100; 50 keeps runtime sane,
                            # raise if the CIs are too wide to decide)
BIN_GRID   = (2, 10, 50)
RUN_UNRESTRICTED = True     # SECONDARY view over all classes, reported separately
N_SPLITS_SECONDARY = 10     # the all-1081-class view costs ~12 s/split (vs 0.3 s for the
                            # abundant subset), so 50 splits would be ~20 min for a view that
                            # is expected to be noise-dominated anyway. Fewer splits => WIDER
                            # CIs there; that is acceptable because no verdict rests on it.
SEED = 42
EMB_ROOT = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}'
# =======================================================================
print('EMB_ROOT =', EMB_ROOT, '| alphas', ALPHAS, '| min cal/class', MIN_CAL_PER_CLASS)


## 2. Mount Drive + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Load OUR scores + report the class-count regime


In [ ]:
import numpy as np
from pcc.data.load import load_split, split_provenance, per_class_counts
from pcc.data.ltc_datasets import NUM_CLASSES

prov = split_provenance(EMB_ROOT, CAL_SPLIT)
print('scores_source:', prov.get('scores_source'))
print('under_gate_exception:', prov.get('under_gate_exception'))
d = load_split(EMB_ROOT, CAL_SPLIT, keys=['logits','labels'])
logits = np.asarray(d['logits'], np.float64)
labels = np.asarray(d['labels']).astype(int)
K = NUM_CLASSES[DATASET]
n = len(labels)
counts = per_class_counts(labels, K)
print(f'{CAL_SPLIT}: {n} samples, {K} classes, accuracy '
      f'{float((logits.argmax(1)==labels).mean()):.4f}')
nz = counts[counts>0]
print('cal samples/class percentiles:',
      {f'p{q}': int(np.percentile(nz,q)) for q in (0,25,50,75,90,100)})
ABUNDANT = np.where(counts >= MIN_CAL_PER_CLASS)[0]
print(f'ABUNDANT classes (>= {MIN_CAL_PER_CLASS} cal samples): {len(ABUNDANT)}/{K}'
      f'  covering {counts[ABUNDANT].sum()}/{n} samples'
      f' ({100*counts[ABUNDANT].sum()/n:.1f}%)')
assert len(ABUNDANT) >= 10, 'too few abundant classes to run the primary analysis'


## 4. PRIMARY - decomposition on the abundant-class label space


In [ ]:
from pcc.eval import decomposition as dc
from pcc.eval.conformal import restrict_to_classes
from pcc.eval.stats import mean_ci
import numpy as np

MECHS = ['temperature'] + [f'energy_b{b}' for b in BIN_GRID] + ['class']

def run_decomposition(class_subset, tag, n_splits=None):
    n_splits = N_SPLITS if n_splits is None else n_splits
    lg, lab, Ksub, _ = restrict_to_classes(logits, labels, class_subset)
    out = {}
    for alpha in ALPHAS:
        acc = {m: [] for m in MECHS}; gsize = []
        rng = np.random.default_rng(SEED)
        for _ in range(n_splits):
            idx = rng.permutation(len(lab)); cal, ev = idx[:len(lab)//2], idx[len(lab)//2:]
            r = dc.phase0_cc_decomposition(lg, lab, Ksub, alpha, cal, ev,
                                           bin_grid=BIN_GRID, estimator='empirical')
            gsize.append(r['global']['avg_set_size'])
            for m in MECHS: acc[m].append(r[m]['gap_vs_global'])
        out[alpha] = {m: mean_ci(v) for m, v in acc.items()}
        out[alpha]['_global_size'] = mean_ci(gsize)
        print(f'--- [{tag}] alpha={alpha}  K={Ksub}  n={len(lab)}  '
              f'global size {np.mean(gsize):.2f} ---')
        for m in MECHS:
            v = out[alpha][m]
            print(f"  {m:16s} gap={v['mean']:+9.3f}  95% CI [{v['ci_low']:+.3f}, {v['ci_high']:+.3f}]")
    return out, Ksub, len(lab)

primary, K_prim, n_prim = run_decomposition(ABUNDANT, 'PRIMARY abundant')


## 5. SECONDARY - all classes (expected to be noise-dominated; never merged)


In [ ]:
secondary = None
if RUN_UNRESTRICTED:
    secondary, K_sec, n_sec = run_decomposition(np.where(counts > 0)[0], 'SECONDARY all',
                                                n_splits=N_SPLITS_SECONDARY)
    print()
    print(f'(secondary used {N_SPLITS_SECONDARY} splits, so its CIs are wider by design)')
else:
    print('skipped')


## 6. Gate verdict on the PRIMARY analysis (Sec 5)


In [ ]:
verdicts = {}
for alpha in ALPHAS:
    r = primary[alpha]
    cls = r['class']
    rivals = {m: r[m] for m in MECHS if m != 'class'}
    best = max(rivals, key=lambda k: rivals[k]['mean'])
    non_overlap = cls['ci_low'] > rivals[best]['ci_high']
    verdicts[str(alpha)] = {'class_gap': cls['mean'], 'class_ci': [cls['ci_low'], cls['ci_high']],
                            'best_rival': best, 'rival_gap': rivals[best]['mean'],
                            'rival_ci': [rivals[best]['ci_low'], rivals[best]['ci_high']],
                            'non_overlapping_CI': bool(non_overlap),
                            'pass': bool(non_overlap and cls['mean'] > rivals[best]['mean'])}
    v = verdicts[str(alpha)]
    print(f"alpha={alpha}: class={v['class_gap']:+.3f} {v['class_ci']} vs "
          f"{v['best_rival']}={v['rival_gap']:+.3f} {v['rival_ci']} -> "
          f"{'PASS' if v['pass'] else 'FAIL'}")

overall = 'PASS' if all(v['pass'] for v in verdicts.values()) else 'FAIL'
print()
print('PHASE 0 GATE (primary, abundant classes):', overall)
if overall != 'PASS':
    print('Sec 5: if the class mechanism does not dominate, the hypothesis that the')
    print('structure lives at the class level does not hold on real data -> report and STOP.')


## 7. Write report


In [ ]:
import time
from pcc.utils.io import write_report

def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return float(o)
    if isinstance(o, np.ndarray): return None
    return o

report = write_report('pcc/reports', f'03_phase0_decomposition_{DATASET}',
    hypothesis='on REAL logits, a per-CLASS offset closes a substantially larger efficiency gap '
               'than a global temperature or a per-sample energy-indexed offset',
    pass_criteria='class gap > best rival AND non-overlapping 95% CIs at every alpha, on the '
                  'PRIMARY abundant-class analysis (Sec 5 fits the per-class offset on abundant '
                  'data, not a realistic budget). Metric = avg set size at worst-class coverage '
                  '>= 1-alpha with out-of-sample per-group estimation (Amendment 3). The '
                  'unrestricted run is SECONDARY and never merged with the primary.',
    config=dict(dataset=DATASET, backbone=BACKBONE, cal_split=CAL_SPLIT,
                alphas=list(ALPHAS), min_cal_per_class=MIN_CAL_PER_CLASS,
                n_splits=N_SPLITS, n_splits_secondary=N_SPLITS_SECONDARY,
                bin_grid=list(BIN_GRID),
                n_abundant_classes=int(len(ABUNDANT)), n_classes_total=int(K),
                scores_source=prov.get('scores_source'),
                under_gate_exception=bool(prov.get('under_gate_exception')),
                amendments=['protocol_amendments.md#amendment-3']),
    seed=SEED,
    results={'primary_abundant': clean(primary), 'verdicts': verdicts,
             'secondary_all_classes': clean(secondary),
             'cal_counts_percentiles': {f'p{q}': int(np.percentile(nz,q))
                                        for q in (0,25,50,75,90,100)}},
    conclusion=f'{overall} (primary: {len(ABUNDANT)} abundant classes of {K})',
    started_at=time.time())
print('report:', report)
print()
print('REMINDER: scores are OURS (gate exception) - cite reports/phase0_checkpoint_gate.md.')
